# Tool-Discovery Agent + MCP Server on Colab

完整流程：PubMed 搜索 → 提取 GitHub 工具 → 注册到 MCP Server → 验证可用

In [ ]:
!pip install -q requests beautifulsoup4 pyyaml pandas tabulate python-dateutil

In [ ]:
import os, subprocess, time, sys, json

REPO_DIR = "/content/scientific_training2_cxy"
if not os.path.exists(REPO_DIR):
    !cd /content && git clone https://github.com/caixiaoyao2025/scientific_training2_cxy.git

os.chdir(REPO_DIR)
os.makedirs("data", exist_ok=True)
print(f"Working in: {os.getcwd()}")

## Step 1: Start MCP Server (HTTP)

In [ ]:
env = os.environ.copy()
env["MCP_DATA_ROOT"] = os.path.join(os.getcwd(), "data")
env["MCP_APP_ROOT"] = os.getcwd()
env["MCP_TRANSPORT"] = "streamable-http"
env["MCP_HOST"] = "0.0.0.0"
env["MCP_PORT"] = "8765"
env["MCP_PATH"] = "/mcp"

LOG = os.path.join(os.getcwd(), "server.log")
log_f = open(LOG, "w")
server_proc = subprocess.Popen(
    [sys.executable, "-u", "server.py"],
    env=env, stdout=log_f, stderr=log_f
)
time.sleep(5)

if server_proc.poll() is not None:
    print(f"Server CRASHED (exit code {server_proc.poll()})")
    log_f.close()
    with open(LOG) as f:
        print(f.read()[-2000:])
else:
    print(f"Server running PID={server_proc.pid}")

## Step 2: Verify MCP Server

In [ ]:
import requests

MCP_URL = "http://127.0.0.1:8765/mcp"

def mcp_call(method, params=None, req_id=1):
    payload = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params or {}}
    r = requests.post(MCP_URL, json=payload, timeout=15)
    return r.json()

# Initialize
init = mcp_call("initialize", {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "colab", "version": "1.0"}
}, req_id=0)
print(f"Init: {init.get('result', {}).get('serverInfo', {})}")

# Send initialized notification
requests.post(MCP_URL, json={"jsonrpc": "2.0", "method": "notifications/initialized"}, timeout=5)

# List tools
result = mcp_call("tools/list")
tools = result.get("result", {}).get("tools", [])
print(f"\n=== {len(tools)} tools loaded ===")
for t in tools:
    print(f"  {t['name']}: {t.get('description', '')[:70]}")

## Step 3: Call a Tool

In [ ]:
# Call list_registered_tools
result = mcp_call("tools/call", {"name": "list_registered_tools", "arguments": {}})
content = result.get("result", {}).get("content", [{}])[0].get("text", "")
print(json.dumps(json.loads(content), indent=2, ensure_ascii=False)[:2000])

## Step 4: Run Discovery Pipeline

In [ ]:
from run_pipeline import run_full_pipeline
run_full_pipeline(query="bioinformatics software tool github", max_results=8)

## Step 5: Re-verify with Newly Discovered Tools

In [ ]:
result = mcp_call("tools/list")
tools = result.get("result", {}).get("tools", [])
print(f"=== {len(tools)} tools after pipeline ===")
for t in tools:
    print(f"  {t['name']}: {t.get('description', '')[:70]}")

## Step 6: Stop Server

In [ ]:
server_proc.terminate()
server_proc.wait()
print("Server stopped.")